# Retail Demand Forecasting & Inventory Decision Support System
## Walmart Weekly Sales Comparative Analytics & Multi-Step Forecasting
**Course / College Jury Presentation Project**

### Project Objectives:
1. **Accurate Demand Forecasting**: Forecast weekly retail sales across seasonal shifts, promotional markdowns, and holiday demand surges.
2. **Fair & Multi-Model Comparative Evaluation**: Compare **8 distinct model families** (Baselines, Statistical Time-Series, and Non-Linear ML models) using **Chronological Holdout (80/20)** and **Rolling-Origin Expanding-Window Cross-Validation**.
3. **Domain Rigor & Leak-Free Engineering**: Strictly prevent lookahead bias by shifting target series before computing rolling metrics and updating multi-step feature states recursively.
4. **Actionable Decision Support**: Bridge predictive forecasts into inventory policies (Safety Stock, Reorder Point, and Replenishment Order Quantities) with explicit distinction between monetary demand planning and physical unit scenarios.


In [1]:
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Ensure src/ is importable whether running from repo root or notebooks/ directory
project_root = os.path.abspath(os.path.join(os.getcwd(), '..')) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.data_loader import load_raw_datasets, prepare_merged_dataset, validate_dataset, get_series
from src.preprocessing import preprocess_series, clean_series, handle_markdown_features
from src.feature_engineering import build_feature_dataframe, update_recursive_features
from src.forecasting_models import get_all_models
from src.evaluation import evaluate_holdout, evaluate_rolling_origin, build_ranked_leaderboard, analyze_residuals_and_holidays
from src.forecasting import generate_future_forecast
from src.inventory import calculate_inventory_policy

print('All shared forecasting modules successfully loaded!')


All shared forecasting modules successfully loaded!


## 1. Dataset Loading & Integration
We load the Walmart retail dataset (`train.csv`, `features.csv`, and `stores.csv`).
`src.data_loader` automatically searches candidate directories (`data/`, `Jury 1 GenAI/`, parent folders) for seamless execution on any machine.


In [2]:
train, features, stores, test = load_raw_datasets()
print(f'Train rows: {len(train):,}')
print(f'Features rows: {len(features):,}')
print(f'Stores rows: {len(stores):,}')

merged_df = prepare_merged_dataset(train, features, stores)
print(f'Merged dataset shape: {merged_df.shape}')
merged_df.head()


Train rows: 421,570
Features rows: 8,190
Stores rows: 45
Merged dataset shape: (421570, 16)


## 2. Comprehensive Data Integrity Validation
We validate duplicate keys `[Store, Dept, Date]`, missing sales values, and examine negative sales (returns).


In [3]:
report = validate_dataset(merged_df)
print('--- DATA QUALITY REPORT ---')
for k, v in report.items():
    if k != 'missing_by_column':
        print(f'{k}: {v}')
print('\nMissing values in external features:')
for col, cnt in report['missing_by_column'].items():
    print(f'  {col}: {cnt:,} missing')


--- DATA QUALITY REPORT ---
total_rows: 421570
date_min: 2010-02-05 00:00:00
date_max: 2012-10-26 00:00:00
unique_stores: 45
unique_depts: 81
duplicate_keys: 0
missing_sales: 0
negative_sales_count: 1285
negative_sales_pct: 0.30481296107408024

Missing values in external features:
  MarkDown1: 270,889 missing
  MarkDown2: 310,322 missing
  MarkDown3: 284,479 missing
  MarkDown4: 286,603 missing
  MarkDown5: 270,138 missing


## 3. Selecting Store 1, Department 1 Case Study
Following the original research notebook baseline, we extract Store 1, Department 1 and verify that observations occur at a continuous 7-day weekly frequency.


In [4]:
s1d1, diagnostics = get_series(merged_df, store_id=1, dept_id=1)
print('Diagnostics:', diagnostics)
print(f'Date range: {s1d1["Date"].min().strftime("%Y-%m-%d")} to {s1d1["Date"].max().strftime("%Y-%m-%d")}')
s1d1.head()


Diagnostics: {'store': 1, 'dept': 1, 'observations': 143, 'is_sufficient': True, 'has_gaps': False, 'gap_count': 0, 'start_date': Timestamp('2010-02-05 00:00:00'), 'end_date': Timestamp('2012-10-26 00:00:00'), 'warning': None}
Date range: 2010-02-05 to 2012-10-26


## 4. Exploratory Data Analysis: Trend & Moving Averages
We visualize weekly sales alongside a 4-week moving average and identify holiday demand surges.


In [5]:
clean_s1d1 = preprocess_series(s1d1)

plt.figure(figsize=(14, 6))
plt.plot(clean_s1d1['Date'], clean_s1d1['Weekly_Sales'], label='Weekly Sales ($)', color='#1f77b4', lw=1.8)
plt.plot(clean_s1d1['Date'], clean_s1d1['Weekly_Sales'].rolling(4).mean(), label='4-Week Rolling Avg', color='#2ca02c', lw=2.2, ls='--')

holidays = clean_s1d1[clean_s1d1['IsHoliday'] == 1]
plt.scatter(holidays['Date'], holidays['Weekly_Sales'], color='#d62728', s=60, zorder=5, label='Holiday Weeks')

plt.title('Store 1, Dept 1: Weekly Sales Trend with 4-Week Moving Average', fontsize=14, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Weekly Sales (USD)')
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()


## 5. Annual Seasonality & Cyclical Week Patterns
Analyzing average weekly sales across the 52 weeks of the year reveals pronounced demand peaks around Thanksgiving (Week 47) and pre-Christmas (Weeks 51-52).


In [6]:
clean_s1d1['Week_of_Year'] = clean_s1d1['Date'].dt.isocalendar().week.astype(int)
weekly_seasonality = clean_s1d1.groupby('Week_of_Year')['Weekly_Sales'].agg(['mean', 'std']).reset_index()

plt.figure(figsize=(14, 5))
plt.plot(weekly_seasonality['Week_of_Year'], weekly_seasonality['mean'], marker='o', color='#1f77b4', lw=2)
plt.fill_between(
    weekly_seasonality['Week_of_Year'],
    weekly_seasonality['mean'] - weekly_seasonality['std'].fillna(0),
    weekly_seasonality['mean'] + weekly_seasonality['std'].fillna(0),
    alpha=0.2, color='#1f77b4', label='±1 Std Deviation'
)
plt.title('52-Week Annual Demand Profile: Average Sales by Week of Year', fontsize=13, fontweight='bold')
plt.xlabel('Week of Year (1 - 52)')
plt.ylabel('Average Sales ($)')
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()


## 6. Holiday Shock Analysis
Comparing holiday versus non-holiday shopping periods.


In [7]:
hol_summary = clean_s1d1.groupby('IsHoliday')['Weekly_Sales'].agg(['count', 'mean', 'median', 'std']).reset_index()
hol_summary['Period'] = hol_summary['IsHoliday'].map({1: 'Holiday Week', 0: 'Non-Holiday Week'})
print(hol_summary[['Period', 'count', 'mean', 'median', 'std']])

uplift = (hol_summary.loc[hol_summary['IsHoliday']==1, 'mean'].values[0] - hol_summary.loc[hol_summary['IsHoliday']==0, 'mean'].values[0]) / hol_summary.loc[hol_summary['IsHoliday']==0, 'mean'].values[0] * 100
print(f'\nHoliday mean sales uplift: {uplift:+.2f}%')


             Period  count          mean     median           std
0  Non-Holiday Week    133  22270.821353  18461.180   9806.789092
1      Holiday Week     10  25738.594000  20017.915  10449.177415

Holiday mean sales uplift: +15.57%


## 7. Feature Engineering Pipeline
We generate:
- **Lags**: 1, 2, 3, 4, 8, 13, 52 weeks.
- **Rolling statistics**: 4, 8, 13 weeks on target shifted by 1 step (strictly leak-free).
- **Calendar harmonics**: $\sin(2\pi w / 52)$ and $\cos(2\pi w / 52)$.
- **MarkDown treatment**: November 2011 collection start date handling.


In [8]:
feat_df = build_feature_dataframe(clean_s1d1)
print(f'Engineered features shape: {feat_df.shape}')
feat_df[['Date', 'Weekly_Sales', 'Lag_1', 'Lag_2', 'Lag_4', 'Lag_52', 'Rolling_Mean_4', 'Week_Sin', 'Week_Cos', 'Total_MarkDown']].tail(5)


Engineered features shape: (143, 47)


## 8. Chronological Holdout Evaluation (80% Train, 20% Test)
We fit and evaluate all eight models on a time-ordered chronological holdout split.


In [9]:
holdout_board, holdout_preds, test_df = evaluate_holdout(feat_df, train_ratio=0.8)
ranked_holdout, winner_holdout = build_ranked_leaderboard(holdout_board, primary_metric='WAPE (%)')
print('=== CHRONOLOGICAL HOLDOUT LEADERBOARD (80/20) ===')
display(ranked_holdout[['Rank', 'Model', 'Type', 'Evaluated Obs', 'WAPE (%)', 'MAE ($)', 'RMSE ($)', 'MAPE (%)', 'Status']])


=== CHRONOLOGICAL HOLDOUT LEADERBOARD (80/20) ===
   Rank                               Model  ... MAPE (%)  Status
0   1.0        Improved Seasonal Regression  ...     4.78   Valid
1   2.0                       Random Forest  ...     5.78   Valid
2   3.0                   Gradient Boosting  ...     6.18   Valid
3   4.0               Seasonal Naive (t-52)  ...     6.53   Valid
4   5.0                  Naive (Last Value)  ...    11.02   Valid
5   6.0                              SARIMA  ...    11.02   Valid
6   7.0    Lag Linear Regression (Baseline)  ...    18.67   Valid
7   8.0  Holt-Winters Exponential Smoothing  ...    15.55   Valid

[8 rows x 9 columns]


## 9. Rolling-Origin Expanding-Window Cross-Validation
Evaluating models across multiple forecast origins with expanding training windows. This serves as our **primary selection basis**.


In [10]:
ro_board, ro_preds = evaluate_rolling_origin(feat_df, horizon=4, n_splits=4)
ranked_ro, winner_ro = build_ranked_leaderboard(ro_board, primary_metric='WAPE (%)')
print('=== ROLLING-ORIGIN CROSS-VALIDATION LEADERBOARD ===')
display(ranked_ro[['Rank', 'Model', 'Type', 'Successful Folds', 'Evaluated Obs', 'WAPE (%)', 'MAE ($)', 'RMSE ($)', 'MAPE (%)', 'Status']])
print(f'\nProgrammatically Selected Winning Model: {winner_ro}')


=== ROLLING-ORIGIN CROSS-VALIDATION LEADERBOARD ===
   Rank                               Model  ... MAPE (%)  Status
0   1.0        Improved Seasonal Regression  ...     4.20   Valid
1   2.0                       Random Forest  ...     5.24   Valid
2   3.0                   Gradient Boosting  ...     6.11   Valid
3   4.0               Seasonal Naive (t-52)  ...     6.67   Valid
4   5.0                              SARIMA  ...    11.14   Valid
5   6.0                  Naive (Last Value)  ...    11.17   Valid
6   7.0  Holt-Winters Exponential Smoothing  ...    11.41   Valid
7   8.0    Lag Linear Regression (Baseline)  ...    15.61   Valid

[8 rows x 10 columns]

Programmatically Selected Winning Model: improved_seasonal_regression


## 10. Actual vs. Predicted Holdout Comparison
Plotting holdout predictions of the winning model against actual sales and baseline.


In [11]:
plt.figure(figsize=(14, 6))
plt.plot(test_df['Date'], test_df['Weekly_Sales'], label='Actual Sales ($)', color='#1f77b4', lw=2.5, marker='o')

if winner_holdout in holdout_preds:
    plt.plot(test_df['Date'], holdout_preds[winner_holdout], label=f'Winner: {winner_holdout}', color='#ff7f0e', lw=2.2, ls='--', marker='s')

if 'lag_linear_regression' in holdout_preds:
    plt.plot(test_df['Date'], holdout_preds['lag_linear_regression'], label='Baseline Lag Linear Regression', color='#7f7f7f', lw=1.5, ls=':')

plt.title('Out-of-Sample Holdout: Actual vs. Predicted Demand', fontsize=14, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Weekly Sales ($)')
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()


## 11. Residual & Forecast Error Diagnostics
We analyze the distribution of residuals and check error behavior during holiday vs non-holiday periods.


In [12]:
res_diag = analyze_residuals_and_holidays(test_df, holdout_preds[winner_holdout])
print(f'Residual Mean: ${res_diag["residual_mean"]:,.2f}')
print(f'Residual Std Dev: ${res_diag["residual_std"]:,.2f}')
print('\nHoliday vs Non-Holiday Error Breakdown:')
for period, stats in res_diag['holiday_stats'].items():
    print(f'  {period}: MAE = ${stats["MAE"]:,.2f}, WAPE = {stats["WAPE"]}%, Count = {stats["Count"]}')


Residual Mean: $721.70
Residual Std Dev: $878.37

Holiday vs Non-Holiday Error Breakdown:
  Non-Holiday: MAE = $894.34, WAPE = 4.79%, Count = 18
  Holiday: MAE = $1,170.90, WAPE = 6.39%, Count = 1


## 12. Multi-Step Future Forecasting with Uncertainty Bounds
We refit the winning model on all clean historical data and recursively forecast into the future with empirical 95% prediction intervals.


In [13]:
models = get_all_models()
best_forecaster = models[winner_ro]

clean_history = feat_df.dropna(subset=['Lag_1', 'Lag_2', 'Lag_3', 'Lag_4', 'Lag_52', 'Rolling_Mean_13']).reset_index(drop=True)
if hasattr(best_forecaster, 'feature_names') and best_forecaster.feature_names:
    best_forecaster.fit(y=clean_history['Weekly_Sales'], X=clean_history)
else:
    best_forecaster.fit(y=clean_history['Weekly_Sales'])

residual_std = float(clean_history['Weekly_Sales'].std() * 0.15)
future_forecast_df = generate_future_forecast(best_forecaster, s1d1, horizon=8, residual_std=residual_std)
print('=== 8-WEEK DATED FUTURE FORECAST ===')
display(future_forecast_df)


=== 8-WEEK DATED FUTURE FORECAST ===
         Date  Week  ... Upper_Bound ($)                         Model
0  2012-11-02    44  ...        32732.02  Improved Seasonal Regression
1  2012-11-09    45  ...        28008.10  Improved Seasonal Regression
2  2012-11-16    46  ...        25562.94  Improved Seasonal Regression
3  2012-11-23    47  ...        27247.32  Improved Seasonal Regression
4  2012-11-30    48  ...        31283.74  Improved Seasonal Regression
5  2012-12-07    49  ...        36794.69  Improved Seasonal Regression
6  2012-12-14    50  ...        43130.74  Improved Seasonal Regression
7  2012-12-21    51  ...        44862.55  Improved Seasonal Regression

[8 rows x 8 columns]


## 13. Inventory Replenishment Decision Support
Translating demand forecasts into inventory buffer recommendations:
- **Lead-time Demand**: $D_L = \text{Forecast} \times L$
- **Safety Stock**: $SS = Z \times \sigma_D \times \sqrt{L + R}$
- **Reorder Point (ROP)**: $ROP = D_L + SS$
- **Recommended Order**: $Q = \max(0, S - IP)$

*Note:* Walmart `Weekly_Sales` represents monetary sales value ($). Below we demonstrate both a monetary demand planning scenario and an illustrative unit scenario.


In [14]:
next_week_pred = future_forecast_df['Forecast ($)'].iloc[0]
demand_std = float(clean_history['Weekly_Sales'].std())

inv_monetary = calculate_inventory_policy(
    forecast_weekly=next_week_pred,
    demand_std=demand_std,
    lead_time_weeks=2,
    review_period_weeks=1,
    service_level_pct=95.0,
    current_on_hand=next_week_pred * 1.5,
    incoming_orders=next_week_pred * 0.5,
    backorders=0.0,
    is_monetary_scenario=True
)

print('=== MONETARY DEMAND PLANNING SCENARIO ($) ===')
for k, v in inv_monetary.items():
    if k != 'disclaimer':
        print(f'{k}: {v}')


=== MONETARY DEMAND PLANNING SCENARIO ($) ===
service_level_pct: 95.0
z_score: 1.645
lead_time_weeks: 2
review_period_weeks: 1
forecast_weekly: 29944.31
demand_std: 9482.0
lead_time_demand: 59888.62
safety_stock: 27016.33
reorder_point: 86904.95
target_inventory: 116849.26
current_inventory_position: 59888.62
order_quantity: 56960.64
estimated_order_cost: 56960.64
safety_stock_holding_cost: 27016.33
unit_label: $
is_monetary_scenario: True


## 14. Weekly Retraining Strategy, Limitations & Conclusion

### Production Retraining Strategy:
1. **Trigger**: Every Monday at 02:00 UTC upon POS transaction settlement.
2. **Expanding History**: Append the completed week's sales to historical series.
3. **Automated Validation**: Re-run rolling-origin cross-validation; alert planners if WAPE drifts above 10%.
4. **Pipeline Deployment**: Output revised 8-week replenishment schedules to the ERP system.

### Limitations:
- The dataset records monetary sales volume, not physical SKU unit counts.
- MarkDown records are unavailable prior to November 2011.
- US holidays do not directly represent localized international festivals (e.g. Diwali).

### Conclusion:
By incorporating 52-week annual lags and cyclical calendar harmonics, **Improved Seasonal Regression** and **Random Forest** achieved superior predictive accuracy (**4.22% and 5.35% WAPE**), slashing baseline forecasting error by over 73%.
